# jA_03 - non-IID sweep near-domain (JURNAL, Konfig A)

Versi **near-domain (76 fitur)** dari R2 (sweep Dirichlet alpha). Di ISRITI, R2 diuji
pada ruang 9-fitur cross-domain (CIC2018 & UNSW). Di sini kita ulang sweep yang SAMA
pada **CIC-IDS2017 dan CSE-CIC-IDS2018 di ruang 76-fitur** agar bisa dibandingkan
apples-to-apples: apakah 'non-IID menurunkan FL' juga berlaku saat domain dekat dan
fiturnya kaya?

**Protokol identik s05:** satu dataset dipecah K=4 klien via Dirichlet(alpha); alpha
turun dari ~IID (100) ke ekstrem (0.05); ukur MCC global pada test dataset itu.
FedAvg + MLP + z-score identik node EC2. Simulasi di SageMaker (banyak kombinasi
alpha x dataset x seed -> tak praktis deploy EC2). Konsisten dgn catatan metodologi R2.

**Prasyarat:** `jA_01` (partisi near-domain di S3).
**Output:** `journal_out/r2near_noniid_{cic2017,cic2018}.csv` + upload S3
`federated_journal/neardomain/results/`.

> Jalankan di SageMaker. Sel 4 menjalankan KEDUA dataset sekaligus. Angka nyata.

## Sel 1 - Setup + util (FedAvg/MLP/Dirichlet, identik node EC2)

In [ ]:
import importlib, sys, subprocess
for m in ('numpy','scikit-learn','boto3'):
    if importlib.util.find_spec(m.replace('scikit-learn','sklearn')) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',m],check=True)
import os, io, csv, numpy as np, boto3
from sklearn.metrics import matthews_corrcoef

BUCKET='ssh-detection-features-232032302717'; REGION='ap-southeast-1'
BASE='federated_journal/neardomain'
OUTDIR='journal_out'; os.makedirs(OUTDIR,exist_ok=True)
s3=boto3.client('s3',region_name=REGION)

# Parameter sweep SAMA dgn s05 (apples-to-apples dgn R2 cross-domain)
ALPHAS=[100,10,1,0.5,0.1,0.05]; K=4; ROUNDS=50; LOCAL_EPOCHS=5; LR=3e-3; SEEDS=[13,42,101]

def load_npz_s3(key):
    obj=s3.get_object(Bucket=BUCKET,Key=key)
    return np.load(io.BytesIO(obj['Body'].read()),allow_pickle=True)

relu=lambda x:np.maximum(0,x); sig=lambda x:1/(1+np.exp(-np.clip(x,-30,30)))
def init_w(d,seed):
    rng=np.random.RandomState(seed); L=[d,128,64,1]; W=[]
    for a,b in zip(L[:-1],L[1:]): W.append(rng.randn(a,b)*np.sqrt(2/a)); W.append(np.zeros(b))
    return W
def fwd(W,X):
    a1=X@W[0]+W[1]; z1=relu(a1); a2=z1@W[2]+W[3]; z2=relu(a2); a3=z2@W[4]+W[5]
    return sig(a3).ravel(),(X,a1,z1,a2,z2)
def train_local(W,X,y,epochs,lr,seed):
    rng=np.random.RandomState(seed); n=len(y); W=[w.copy() for w in W]
    for _ in range(epochs):
        idx=rng.permutation(n)
        for s in range(0,n,256):
            bi=idx[s:s+256]; Xb=X[bi]; yb=y[bi].astype(float)
            o,(X_,a1,z1,a2,z2)=fwd(W,Xb); g=(o-yb)/max(len(yb),1)
            gW4=z2.T@g[:,None]; gb4=g.sum(0); d2=(g[:,None]@W[4].T)*(a2>0)
            gW2=z1.T@d2; gb2=d2.sum(0); d1=(d2@W[2].T)*(a1>0); gW0=X_.T@d1; gb0=d1.sum(0)
            for i,gv in enumerate([gW0,gb0,gW2,gb2,gW4,gb4]):
                W[i]=W[i]-lr*np.asarray(gv).reshape(W[i].shape)
    return W
def fedavg(Ws,ns):
    tot=sum(ns); return [sum((ns[k]/tot)*Ws[k][i] for k in range(len(Ws))) for i in range(len(Ws[0]))]
def zfit(X): m=X.mean(0); s=X.std(0); s[s==0]=1; return m,s
def dirichlet_partition(y,K,alpha,seed):
    rng=np.random.RandomState(seed); idx_by_c={c:np.where(y==c)[0] for c in np.unique(y)}
    client_idx=[[] for _ in range(K)]
    for c,idc in idx_by_c.items():
        rng.shuffle(idc); props=rng.dirichlet([alpha]*K); cuts=(np.cumsum(props)*len(idc)).astype(int)[:-1]
        for k,part in enumerate(np.split(idc,cuts)): client_idx[k].extend(part.tolist())
    return [np.array(c) for c in client_idx if len(c)>0]
print('util siap | ALPHAS=',ALPHAS,'K=',K,'ROUNDS=',ROUNDS)

## Sel 2 - Fungsi sweep untuk satu dataset

In [ ]:
def sweep_dataset(cid):
    dtr=load_npz_s3(f'{BASE}/data/{cid}/{cid}_train.npz')
    dte=load_npz_s3(f'{BASE}/data/{cid}/{cid}_test.npz')
    Xtr=np.asarray(dtr['X'],float); ytr=np.asarray(dtr['y'],int)
    Xte=np.asarray(dte['X'],float); yte=np.asarray(dte['y'],int)
    d=Xtr.shape[1]
    m,s=zfit(Xtr); Xtr_z=(Xtr-m)/s; Xte_z=(Xte-m)/s
    print(f'{cid}: train {Xtr.shape} pos={ytr.mean():.3f} | test {Xte.shape} (d={d})')
    def run_fedavg(parts,seed):
        W=init_w(d,seed)
        for t in range(ROUNDS):
            Ws=[]; ns=[]
            for pidx in parts:
                Wl=train_local(W,Xtr_z[pidx],ytr[pidx],LOCAL_EPOCHS,LR,seed); Ws.append(Wl); ns.append(len(pidx))
            W=fedavg(Ws,ns)
        o,_=fwd(W,Xte_z); return matthews_corrcoef(yte,(o>=0.5).astype(int))
    rows=[]
    for a in ALPHAS:
        vals=[]
        for sd in SEEDS:
            parts=dirichlet_partition(ytr,K,a,sd); vals.append(run_fedavg(parts,sd))
        rows.append(dict(dataset=cid,alpha=a,mcc_mean=round(float(np.mean(vals)),4),
                         mcc_std=round(float(np.std(vals)),4),
                         values=';'.join(str(round(v,4)) for v in vals)))
        print(f'  {cid} alpha={a:<5} MCC={rows[-1]["mcc_mean"]:.3f} +/- {rows[-1]["mcc_std"]:.3f}')
    return rows
print('fungsi sweep_dataset siap')

## Sel 3 - Jalankan sweep KEDUA dataset + simpan CSV + upload S3

In [ ]:
import time
for cid in ['cic2017','cic2018']:
    t0=time.time(); rows=sweep_dataset(cid); dt=time.time()-t0
    path=os.path.join(OUTDIR,f'r2near_noniid_{cid}.csv')
    with open(path,'w',newline='') as f:
        w=csv.DictWriter(f,fieldnames=['dataset','alpha','mcc_mean','mcc_std','values']); w.writeheader(); w.writerows(rows)
    try: s3.upload_file(path,BUCKET,f'{BASE}/results/{os.path.basename(path)}'); up='upload OK'
    except Exception as e: up='(upload dilewati: %s)'%e
    print(f'{cid} selesai [{dt:.0f}s] -> {path} | {up}\n')
print('=== jA_03 SELESAI. R2 near-domain untuk CIC2017 & CIC2018 siap. ===')

## Catatan untuk naskah jurnal

- R2 near-domain memakai protokol sweep IDENTIK dgn R2 cross-domain ISRITI
  (K=4, alpha {100..0.05}, 3 seed, 50 ronde) tetapi di ruang 76-fitur near-domain,
  sehingga perbandingan 'efek non-IID: near vs far' adil.
- Simulasi di SageMaker (konsisten dgn R2 ISRITI yang juga simulasi).
- Titik putus = alpha di mana MCC jatuh tajam / std melonjak.
- Angka untuk tabel/kurva jurnal diambil dari r2near_noniid_*.csv (hasil nyata).